In [11]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "wolf2019visually")
original_data_pathway = os.path.join(pathway, "original_data")

# complete_path_1 = os.path.join(original_data_pathway, "Data for Wolf & Tomasello 2019 - Study 1.sav")
complete_path_1 = os.path.join(original_data_pathway, "Study_1_data_long.sav")
complete_path_2 = os.path.join(original_data_pathway, "Study_1_data.sav")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [12]:
import pandas as pd
import numpy as np
import pyreadstat


df2 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)

df = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)

# original_data_pathway_out = os.path.join(original_data_pathway, 'Study_1_data.csv')
# df2.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)


df['study_id']="wolf2019visually"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df2.columns = map(str.lower, df2.columns)
df2=df2.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns

In [13]:
df.rename(columns={"subname": "ape",
    "species": "species_original",
    "group":"group_original",
    "colorupdown":"color_up_down"}, inplace=True)
df['group_original'].replace('bonobo', np.nan, inplace=True)

df['subgroup'] = df['group_original'].str.slice(0,1)


In [14]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')
# df.columns
df.rename(columns={"ape": "participant"}, inplace=True)
df.columns

Index(['participant', 'gender', 'timeses1', 'timeses2', 'timeja', 'timeda',
       'order', 'species_original', 'group_original', 'calfile', 'eyejascreen',
       'eyejaexp', 'eyedascreen', 'eyedaexp', 'jaeyestart', 'daeyestart',
       'study_id', 'subgroup', 'name', 'species', 'sex'],
      dtype='object')

In [15]:
JA = df[df.order.str.contains("ja first")]
j_list_1 =JA[['study_id','participant', 'sex','species','subgroup','order','timeja', 'eyejascreen','eyejaexp']].copy()
j_list_1['session_1']=1
j_list_1['condition_1']='joint_attention'
j_list_2 =JA[['study_id','participant', 'sex','species','subgroup','order','timeda', 'eyedascreen','eyedaexp']].copy()
j_list_2['session_2']=2
j_list_2['condition_2']='disjoint_attention'

DA = df[df.order.str.contains("da first")]
d_list_1 =DA[['study_id','participant', 'sex','species','subgroup','order','timeda', 'eyedascreen','eyedaexp']].copy()
d_list_1['session_1']=1
d_list_1['condition_1']='disjoint_attention'
d_list_2 =DA[['study_id','participant', 'sex','species','subgroup','order', 'timeja','eyejascreen','eyejaexp']].copy()
d_list_2['session_2']=2
d_list_2['condition_2']='joint_attention'


df_temp1 = j_list_1.values.tolist() + j_list_2.values.tolist() + d_list_1.values.tolist() + d_list_2.values.tolist() 

df_new = pd.DataFrame(df_temp1, columns=['study_id','participant', 'sex','species','subgroup','order', 'approach_latency','time_spent_looking_at_screen','time_spent_looking_at_experimenter', 'session','condition'])


In [16]:
df_new['order'].replace(' ', '_', inplace=True, regex=True)
green = ['alex', 'frederike','zira','hope','bangolo','dorien','kofi','lobo','lome','sandra','tai','riet','jasongo','kuno','luiza','joey']
red = ['daza','jeudi','corrie','frodo','robert','fimi','gemena','yasa']

for x in green:
    df_new.loc[df_new.participant == x, ['color_up_down']] = 'green'
for x in red:
    df_new.loc[df_new.participant == x, ['color_up_down']] = 'red'
df_new=df_new.sort_values(by = ['participant','session'])

In [17]:
df_new = df_new.assign(excluded='no') 
exclude_list = ['natascha','fraukje']
for x in exclude_list:
    df_new.loc[df_new.participant == x, ['excluded']] = 'yes'

In [18]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df_new= df_new.merge(subject_list,left_on='participant', right_on='name', how='left')
df_new.rename(columns={"age": "age_in_years", "subgroup":"species_subgroup"}, inplace=True)

In [19]:
wolf2019visually_standardized=df_new[['study_id','participant', 'age_in_years','sex','species','species_subgroup',
                                      'session','condition','order', 
                                      'approach_latency','time_spent_looking_at_screen','time_spent_looking_at_experimenter',
                                      'excluded']]

##removed 'calfile','jaeyestart', 'daeyestart',


In [20]:
comp_out_path_stand = os.path.join(out_pathway, 'wolf2019visually_exp1_standardized.csv')
wolf2019visually_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names = wolf2019visually_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
wolf2019visually_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'wolf2019visually_exp1_glossary.csv')
wolf2019visually_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)